# Haberman Survival Classification

306 breast-cancer surgery patients, each described by just three numbers: age at operation, year of operation, and the number of positive axillary lymph nodes detected. Predict whether the patient survived five years or longer — a deliberately hard, small, imbalanced problem.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Haberman's Survival (id 43)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/43/data.csv')))
print(df.shape)
print('Patients per survival code (1 = survived 5+ years, 2 = died within 5 years):')
print(df['survival_status'].value_counts().sort_index())
df.head(3)


(306, 4)
Patients per survival code (1 = survived 5+ years, 2 = died within 5 years):
survival_status
1    225
2     81
Name: count, dtype: int64


,age,operation_year,positive_auxillary_nodes,survival_status
0,30,64,1,1
1,30,62,3,1
2,30,65,0,1


In [2]:

feats = ['age', 'operation_year', 'positive_auxillary_nodes']
X = df[feats]; y = df['survival_status'].map({1: 'survived', 2: 'died'})
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.6883
Logistic Regression accuracy: 0.7403
              precision    recall  f1-score   support

        died       0.39      0.35      0.37        20
    survived       0.78      0.81      0.79        57

    accuracy                           0.69        77
   macro avg       0.58      0.58      0.58        77
weighted avg       0.68      0.69      0.68        77



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp)
imp.plot(kind='barh', title='Which factor matters most for survival?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


age                         0.437587
positive_auxillary_nodes    0.324976
operation_year              0.237437
dtype: float64


Winner: Logistic Regression
